# 3. Modelos de Visión

Analizar imágenes con modelos multimodales.

## Setup

Instalar SDK.

In [ ]:
%pip install --upgrade openai pillow

## API Key

Usar variable de entorno.

In [ ]:
import os
import base64
from pathlib import Path
from getpass import getpass
from IPython.display import Image, display
from openai import OpenAI

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY: ")

client = OpenAI()

VISION_MODEL = os.getenv("VISION_MODEL", "gpt-4.1-mini")
print(f"Modelo: {VISION_MODEL}")

## 3.1 Estructura de mensajes con imágenes

Un mensaje puede mezclar texto e imagen.

In [ ]:
image_url = "https://api.nga.gov/iiif/a2e6da57-3cd1-4235-b20e-95dcaefed6c8/full/!800,800/0/default.jpg"

response = client.responses.create(
    model=VISION_MODEL,
    input=[{
        "role": "user",
        "content": [
            {"type": "input_text", "text": "Describe la imagen en 3 bullets."},
            {"type": "input_image", "image_url": image_url},
        ],
    }],
)

print(response.output_text)

## 3.2 Formas de proporcionar imágenes

Opción 1: URL pública.

In [ ]:
response = client.responses.create(
    model=VISION_MODEL,
    input=[{
        "role": "user",
        "content": [
            {"type": "input_text", "text": "¿Qué objetos principales aparecen?"},
            {"type": "input_image", "image_url": image_url},
        ],
    }],
)

print(response.output_text)

Opción 2: Base64 local.

In [ ]:
def encode_image(image_path: str) -> str:
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")


local_image_path = Path("imagen_local.jpg")

if local_image_path.exists():
    base64_image = encode_image(str(local_image_path))

    response = client.responses.create(
        model=VISION_MODEL,
        input=[{
            "role": "user",
            "content": [
                {"type": "input_text", "text": "Describe esta imagen."},
                {
                    "type": "input_image",
                    "image_url": f"data:image/jpeg;base64,{base64_image}",
                },
            ],
        }],
    )

    print(response.output_text)
else:
    print("Coloca imagen_local.jpg junto al notebook.")

Opción 3: File ID.

In [ ]:
def upload_image_for_vision(path: str) -> str:
    with open(path, "rb") as f:
        result = client.files.create(
            file=f,
            purpose="vision",
        )
    return result.id


local_image_path = Path("imagen_local.jpg")

if local_image_path.exists():
    file_id = upload_image_for_vision(str(local_image_path))

    response = client.responses.create(
        model=VISION_MODEL,
        input=[{
            "role": "user",
            "content": [
                {"type": "input_text", "text": "Analiza esta imagen."},
                {"type": "input_image", "file_id": file_id},
            ],
        }],
    )

    print(response.output_text)
else:
    print("Coloca imagen_local.jpg junto al notebook.")

## 3.3 Parámetro `detail`

Controla precisión/costo.

In [ ]:
for detail in ["low", "high"]:
    response = client.responses.create(
        model=VISION_MODEL,
        input=[{
            "role": "user",
            "content": [
                {"type": "input_text", "text": f"Describe la imagen. detail={detail}"},
                {
                    "type": "input_image",
                    "image_url": image_url,
                    "detail": detail,
                },
            ],
        }],
    )

    print(f"\n--- detail={detail} ---")
    print(response.output_text)

## 3.4 Modelos compatibles

Usar modelos con visión.

Ejemplos:
- `gpt-4.1-mini`
- `gpt-4.1`
- `gpt-5.5`
- modelos futuros multimodales

In [ ]:
print("Modelo actual:", VISION_MODEL)

## 3.5 Limitaciones

- Puede fallar con texto pequeño.
- No asumir medidas exactas.
- No usar para diagnóstico médico.
- Verificar datos críticos.
- Imágenes cuentan como tokens.

## 3.6 Ejemplo completo: extracción visual

Devolver JSON simple.

In [ ]:
response = client.responses.create(
    model=VISION_MODEL,
    input=[{
        "role": "user",
        "content": [
            {
                "type": "input_text",
                "text": """
                Analiza la imagen y responde solo JSON:
                {
                  "descripcion": "...",
                  "objetos": ["..."],
                  "colores": ["..."],
                  "posible_uso": "..."
                }
                """
            },
            {"type": "input_image", "image_url": image_url, "detail": "high"},
        ],
    }],
)

print(response.output_text)

## 3.7 Casos de uso comunes

- Catálogos.
- Moderación visual.
- OCR ligero.
- Inspección de imágenes.
- Soporte con screenshots.

## 3.8 Buenas prácticas

- Pedir formato exacto.
- Usar `detail="low"` para previews.
- Usar `detail="high"` para análisis.
- Recortar imágenes grandes.
- Validar resultados importantes.

## Ejercicio

Analizar un screenshot propio.

In [ ]:
screenshot_path = Path("screenshot.png")

if screenshot_path.exists():
    base64_image = encode_image(str(screenshot_path))

    response = client.responses.create(
        model=VISION_MODEL,
        input=[{
            "role": "user",
            "content": [
                {"type": "input_text", "text": "Detecta el problema de UI en esta captura. Sé breve."},
                {
                    "type": "input_image",
                    "image_url": f"data:image/png;base64,{base64_image}",
                    "detail": "high",
                },
            ],
        }],
    )

    print(response.output_text)
else:
    print("Coloca screenshot.png junto al notebook.")